# ChronoPDE V2 Phase 4B — integration audit

Attach the private **Phase 3 Development** and **Phase 4 Feasibility Results** datasets, enable Internet, and select one T4 GPU. This notebook performs evaluation only and never accesses confirmatory data.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = 'https://github.com/madhavkapoor13/ChronoPDE.git'
BRANCH = 'codex/chronopde-v2-phase4b'
REPOSITORY = Path('/kaggle/working/ChronoPDE')
if not REPOSITORY.exists():
    clone = ['git', 'clone', '--branch', BRANCH, '--single-branch']
    subprocess.run([*clone, REPOSITORY_URL, str(REPOSITORY)], check=True)
install = [sys.executable, '-m', 'pip', 'install', '--quiet', '-e', str(REPOSITORY)]
subprocess.run(install, check=True)
commit = subprocess.check_output(
    ['git', '-C', str(REPOSITORY), 'rev-parse', 'HEAD'], text=True
).strip()
print('Commit:', commit)
print('GPU:', __import__('torch').cuda.get_device_name(0))

In [ ]:
data_candidates = list(Path('/kaggle/input').rglob('chronopde_v2_development.h5'))
assert len(data_candidates) == 1, data_candidates
DATA = data_candidates[0]
archive_candidates = list(
    Path('/kaggle/input').rglob('chronopde_v2_phase4_outputs.zip')
)
if len(archive_candidates) == 1:
    RESULTS = archive_candidates[0]
else:
    roots = [path for path in Path('/kaggle/input').iterdir() if path.is_dir()]
    expanded = [root for root in roots if list(root.rglob('*-fft-feasibility-*'))]
    assert len(expanded) == 1, (archive_candidates, expanded)
    RESULTS = expanded[0]
print('Development data:', DATA)
print('Phase 4 results:', RESULTS)
check = [
    sys.executable, 'scripts/chronopde_v2.py', 'phase4b',
    '--data-path', str(DATA), '--phase4-results', str(RESULTS),
    '--check-only', '--format', 'json',
]
subprocess.run(check, cwd=REPOSITORY, check=True)

In [ ]:
command = [
    sys.executable, 'scripts/chronopde_v2.py', 'phase4b',
    '--data-path', str(DATA), '--phase4-results', str(RESULTS),
    '--device', 'cuda', '--format', 'json',
]
log_path = Path('/kaggle/working/phase4b_command.log')
with log_path.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(
        command, cwd=REPOSITORY, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
    return_code = process.wait()
packages = list(
    REPOSITORY.rglob('chronopde_v2_phase4b_outputs.zip')
)
destination = Path('/kaggle/working/chronopde_v2_phase4b_outputs.zip')
if len(packages) == 1:
    shutil.copy2(packages[0], destination)
else:
    with __import__('zipfile').ZipFile(destination, 'w') as archive:
        archive.write(log_path, log_path.name)
print('Return code:', return_code)
print('Download:', destination)
assert return_code == 0, 'Audit failed; download the diagnostic ZIP and log.'